# Business Analysis for Data Analysts Bootcamp

A practical introduction to **32 business analysis concepts** with Python examples. Every lesson has two cells: a Markdown explanation followed by code with inline `#` comments.

**Requirements:** Python, NumPy, pandas, SciPy, and Matplotlib. These are commonly available in Anaconda. If you need to install them, run `%pip install numpy pandas scipy matplotlib` in a notebook cell.

The examples use synthetic data and do not require downloads. Run the notebook from top to bottom because several lessons reuse the sample sales dataset initialized in Lesson 1.

## What you'll practice
- Define business problems, SMART objectives, stakeholders, and KPIs.
- Analyze revenue, margin, unit economics, break-even, and ROI.
- Investigate customer funnels, segments, retention, churn, and RFM.
- Map processes, find bottlenecks, assess risks, and prioritize requirements.
- Use scenarios, simple forecasting, A/B testing, and clear business recommendations.

**Note:** This is a business-analysis learning notebook, not a substitute for validating business definitions, stakeholder needs, or source data.

## 1. Start with a Business Problem

Business analysis starts by clarifying the decision, not by opening a chart. A useful problem statement says **what is happening, who is affected, where and when it happens, and why it matters**. Separate the observed symptom from the suspected cause.

A good analysis question is answerable with evidence. Instead of “sales are bad,” ask “Which product categories and channels explain the year-over-year decline in gross profit during the last quarter?”

In [ ]:
# Import and prepare reproducible synthetic sales data for the later lessons.
rng = np.random.default_rng(2026)  # Create a reproducible random generator.
number_of_lines = 900  # Set the number of synthetic transaction lines.
order_dates = pd.to_datetime(rng.integers(pd.Timestamp("2025-01-01").value // 10**9, pd.Timestamp("2026-07-01").value // 10**9, size=number_of_lines), unit="s").normalize()  # Generate order dates between January 2025 and June 2026.
customer_ids = rng.integers(1001, 1151, size=number_of_lines)  # Assign transaction lines to 150 illustrative customers.
categories = rng.choice(["Accessories", "Office", "Technology", "Furniture"], size=number_of_lines, p=[0.25, 0.30, 0.25, 0.20])  # Assign product categories.
channels = rng.choice(["Online", "Store", "Partner"], size=number_of_lines, p=[0.52, 0.33, 0.15])  # Assign sales channels.
regions = rng.choice(["North", "South", "East", "West"], size=number_of_lines)  # Assign geographic regions.
units = rng.integers(1, 6, size=number_of_lines)  # Generate positive quantities for each line.
unit_price_map = {"Accessories": 18.0, "Office": 25.0, "Technology": 110.0, "Furniture": 150.0}  # Define illustrative sales prices by category.
unit_cost_map = {"Accessories": 8.0, "Office": 12.0, "Technology": 72.0, "Furniture": 98.0}  # Define illustrative costs by category.
unit_prices = np.array([unit_price_map[c] for c in categories]) * rng.uniform(0.9, 1.1, size=number_of_lines)  # Add modest variation to selling prices.
unit_costs = np.array([unit_cost_map[c] for c in categories]) * rng.uniform(0.92, 1.08, size=number_of_lines)  # Add modest variation to costs.
line_revenue = units * unit_prices  # Calculate revenue for each transaction line.
line_cost = units * unit_costs  # Calculate cost for each transaction line.
orders = pd.DataFrame({  # Combine generated fields into the reusable transaction table.
    "order_date": order_dates,  # Store the order date.
    "order_id": [f"O{n:05d}" for n in rng.integers(1, 620, size=number_of_lines)],  # Allow multiple lines and repeated order IDs.
    "customer_id": customer_ids,  # Store the customer identifier.
    "category": categories,  # Store the product category.
    "channel": channels,  # Store the sales channel.
    "region": regions,  # Store the region.
    "units": units,  # Store the units sold.
    "unit_price": unit_prices.round(2),  # Store the rounded selling price.
    "unit_cost": unit_costs.round(2),  # Store the rounded unit cost.
    "revenue": line_revenue.round(2),  # Store transaction-line revenue.
    "cost": line_cost.round(2),  # Store transaction-line cost.
})  # Finish building the dataset.
orders["gross_profit"] = orders["revenue"] - orders["cost"]  # Calculate simplified gross profit for each line.
orders["order_month"] = orders["order_date"].dt.to_period("M")  # Add month periods for time analysis.
print(orders.head())  # Preview the synthetic sales dataset.
print("\nRows:", len(orders), "| Unique orders:", orders["order_id"].nunique(), "| Customers:", orders["customer_id"].nunique())  # Show dataset scale and grain.
print("\nNote: order IDs repeat because each row is a transaction line, not necessarily a complete order.")  # Explain an important grain consideration.

## 2. Define SMART Objectives

SMART objectives are **Specific, Measurable, Achievable, Relevant, and Time-bound**. They help translate a broad wish into a testable business target.

A target should be grounded in a baseline, an owner, and an agreed measurement definition. A numeric goal without a clear denominator or deadline can be interpreted in different ways by different teams.

In [ ]:
baseline_conversion = 0.035  # Define the current conversion rate as 3.5%.
target_conversion = 0.040  # Set an example target conversion rate of 4.0%.
deadline = "within 90 days"  # Set the deadline for reaching the target.
relative_lift = (target_conversion - baseline_conversion) / baseline_conversion  # Calculate relative improvement needed.
percentage_point_change = target_conversion - baseline_conversion  # Calculate absolute percentage-point change.
print(f"Baseline conversion: {baseline_conversion:.1%}")  # Display the current performance.
print(f"Target conversion: {target_conversion:.1%}")  # Display the intended performance.
print(f"Required change: {percentage_point_change:.1%} percentage points")  # Display the absolute rate change.
print(f"Relative lift required: {relative_lift:.1%}")  # Display the proportional lift.
print(f"Deadline: {deadline}")  # Display the time limit.

## 3. Identify Stakeholders

Stakeholders are people or groups who influence a business decision or are affected by it. Typical stakeholders include sponsors, operational users, technical teams, customers, compliance teams, and finance.

A stakeholder map makes ownership and communication needs visible. Influence and interest are useful starting dimensions, but avoid treating them as a substitute for direct conversations.

In [ ]:
stakeholders = pd.DataFrame({  # Create a simple illustrative stakeholder register.
    "stakeholder": ["Sales Director", "Operations Lead", "Finance Analyst", "Customer Support", "Data Team"],  # Name stakeholder groups.
    "influence": ["High", "High", "Medium", "Low", "Medium"],  # Rate influence on the decision.
    "interest": ["High", "High", "High", "Medium", "Medium"],  # Rate interest in the outcome.
    "responsibility": ["Sponsor", "Process owner", "Validate financial impact", "Share customer pain points", "Provide data and analysis"],  # State the expected contribution.
})  # Finish the stakeholder register.
print(stakeholders)  # Review the stakeholder list and responsibilities.
print("\nHigh-influence stakeholders:")  # Label a focused view for engagement planning.
print(stakeholders[stakeholders["influence"] == "High"])  # Filter the stakeholders with high influence.

## 4. Build a KPI Dictionary

A Key Performance Indicator (KPI) is a defined measure linked to a business objective. Every KPI should have a name, purpose, formula, unit, data source, owner, frequency, and caveats.

Metric definitions prevent teams from comparing different calculations under the same label. Be especially explicit about denominators, time zones, cancellations, refunds, and whether the metric is a count or a rate.

In [ ]:
kpi_dictionary = pd.DataFrame([  # Create a small KPI dictionary for a sales team.
    {"KPI": "Net Revenue", "Formula": "Gross sales - discounts - refunds", "Unit": "Currency", "Owner": "Finance", "Frequency": "Weekly"},  # Define a monetary KPI.
    {"KPI": "Conversion Rate", "Formula": "Purchases / eligible visitors", "Unit": "Percent", "Owner": "Growth", "Frequency": "Weekly"},  # Define a rate with its denominator.
    {"KPI": "Gross Margin", "Formula": "(Net revenue - COGS) / net revenue", "Unit": "Percent", "Owner": "Finance", "Frequency": "Monthly"},  # Define a profitability KPI.
    {"KPI": "Repeat Purchase Rate", "Formula": "Customers with 2+ orders / purchasing customers", "Unit": "Percent", "Owner": "CRM", "Frequency": "Monthly"},  # Define a retention KPI.
])  # Finish the dictionary.
print(kpi_dictionary.to_string(index=False))  # Print the definitions in a compact format.

## 5. Create a KPI Tree

A KPI tree breaks a high-level outcome into drivers that teams can influence. For example, revenue can be decomposed into order count × average order value; orders may depend on traffic × conversion rate.

This tree helps connect diagnostic metrics to business actions. Check that the mathematical relationship is valid for the business model and that metrics use compatible time windows and populations.

In [ ]:
visitors = 20000  # Set the number of eligible visitors.
conversion_rate = 0.04  # Set the purchase conversion rate.
orders_per_buyer = 1.2  # Set the average number of orders per buyer for the period.
average_order_value = 75.0  # Set the average monetary value of an order.
buyers = visitors * conversion_rate  # Estimate the number of buyers.
estimated_orders = buyers * orders_per_buyer  # Estimate orders from buyers and frequency.
estimated_revenue = estimated_orders * average_order_value  # Calculate revenue from orders and average order value.
print(f"Visitors: {visitors:,.0f}")  # Display the top-of-funnel input.
print(f"Estimated buyers: {buyers:,.0f}")  # Display buyers implied by conversion.
print(f"Estimated orders: {estimated_orders:,.0f}")  # Display orders implied by purchase frequency.
print(f"Estimated revenue: ${estimated_revenue:,.2f}")  # Display revenue implied by the driver tree.

## 6. Understand the Business Model

A business model explains how an organization creates, delivers, and captures value. For analysis, identify customers, value proposition, channels, revenue streams, major costs, and the unit being sold.

Knowing the model helps choose relevant metrics. A subscription business may focus on recurring revenue and churn; a retailer may focus on margin, inventory turnover, and conversion.

In [ ]:
business_model = pd.DataFrame({  # Map the example model's key components.
    "Component": ["Customer", "Value proposition", "Channel", "Revenue stream", "Major cost", "Core unit"],  # Name business-model elements.
    "Example": ["Small businesses", "Convenient office supplies", "Website and sales reps", "Product sales", "Product cost and delivery", "Order"],  # Fill in an illustrative answer for each.
    "Analysis question": ["Which customers are most valuable?", "Which need is being solved?", "Which channel converts best?", "What drives revenue growth?", "Which costs compress margin?", "What is profitable per order?"],  # Link each element to a question.
})  # Finish the business-model canvas.
print(business_model.to_string(index=False))  # Display the canvas as a compact table.

## 7. SWOT Analysis

SWOT organizes strategic factors into **Strengths, Weaknesses, Opportunities, and Threats**. Strengths and weaknesses are typically internal; opportunities and threats are typically external.

SWOT is a structured discussion tool, not a statistical proof. Ground each item in evidence, state its implication, and assign follow-up actions rather than filling the matrix with unsupported opinions.

In [ ]:
swot = pd.DataFrame({  # Organize example observations into the four SWOT categories.
    "Category": ["Strength", "Weakness", "Opportunity", "Threat"],  # List the four SWOT groups.
    "Observation": ["High repeat purchase rate", "Slow order fulfillment", "Growing demand in a new region", "Competitor discounts increase"],  # Add example observations.
    "Evidence to collect": ["Repeat order data", "Order-to-delivery timestamps", "Market and regional sales data", "Competitor price tracking"],  # Specify how each observation can be checked.
    "Possible response": ["Protect loyalty experience", "Find operational bottlenecks", "Run a controlled regional pilot", "Review price and value proposition"],  # Suggest a response to investigate.
})  # Finish the SWOT table.
print(swot.to_string(index=False))  # Display the SWOT and how to validate it.

## 8. Segment Customers and Markets

Segmentation groups customers by meaningful attributes, behavior, needs, or value. Useful segment dimensions include geography, customer type, acquisition channel, purchase frequency, and order value.

Choose segmentation that changes a decision. Segments should be clearly defined, measurable, useful, and large enough to act on. Avoid creating many tiny groups that only make reporting harder.

In [ ]:
# Create a small segment summary from the reusable transaction dataset.
segment_summary = orders.groupby(["region", "channel"], as_index=False).agg(  # Group records by region and channel.
    revenue=("revenue", "sum"),  # Add revenue within each segment.
    gross_profit=("gross_profit", "sum"),  # Add gross profit within each segment.
    orders=("order_id", "nunique"),  # Count unique orders within each segment.
)  # Finish segment aggregation.
segment_summary["gross_margin"] = segment_summary["gross_profit"] / segment_summary["revenue"]  # Calculate gross margin for each segment.
segment_summary = segment_summary.sort_values("revenue", ascending=False)  # Rank segments from highest to lowest revenue.
print(segment_summary.round(3).to_string(index=False))  # Display the segment performance table.

## 9. Customer Journey and Funnel

A customer journey describes the steps a person takes toward an outcome. A funnel quantifies how many people reach each step and where they leave.

Funnel metrics need consistent population rules: the same time window, user identity, and eligibility rules. If steps are not tracked at the individual level, counts alone may not tell you whether the same people progressed through each step.

In [ ]:
funnel = pd.DataFrame({  # Store example counts at each step of a customer journey.
    "stage": ["Visited site", "Viewed product", "Added to cart", "Started checkout", "Purchased"],  # List funnel stages in order.
    "users": [12000, 7800, 2400, 1600, 960],  # Count eligible users reaching each stage.
})  # Finish the funnel table.
funnel["conversion_from_previous"] = funnel["users"] / funnel["users"].shift(1)  # Calculate step-to-step conversion.
funnel["conversion_from_start"] = funnel["users"] / funnel.loc[0, "users"]  # Calculate conversion relative to the first stage.
funnel["drop_off_from_previous"] = 1 - funnel["conversion_from_previous"]  # Calculate the step-level drop-off rate.
print(funnel.to_string(index=False, formatters={  # Format rates as percentages for readability.
    "conversion_from_previous": lambda x: "—" if pd.isna(x) else f"{x:.1%}",  # Format first-step value and conversions.
    "conversion_from_start": lambda x: f"{x:.1%}",  # Format cumulative conversion.
    "drop_off_from_previous": lambda x: "—" if pd.isna(x) else f"{x:.1%}",  # Format drop-off rates.
}))  # Finish printing the funnel report.

## 10. Revenue Drivers and Growth

Revenue should be decomposed into drivers rather than viewed only as a total. Depending on the business, useful drivers include active customers, purchase frequency, average order value, pricing, product mix, and returns.

Growth can be expressed as an absolute change or a relative percentage change. Always compare like-for-like periods and beware of partial months, seasonality, and one-off events.

In [ ]:
monthly_revenue = pd.Series(  # Create an illustrative monthly revenue series.
    [82000, 85000, 87000, 91000, 90000, 98000],  # Store revenue from January through June.
    index=pd.period_range("2026-01", periods=6, freq="M"),  # Label each observation with its month.
    name="revenue",  # Give the series a meaningful name.
)  # Finish the series.
revenue_table = monthly_revenue.to_frame()  # Convert the series to a DataFrame for reporting.
revenue_table["absolute_change"] = revenue_table["revenue"].diff()  # Calculate month-over-month currency change.
revenue_table["growth_rate"] = revenue_table["revenue"].pct_change()  # Calculate month-over-month growth rate.
print(revenue_table.round(3))  # Display revenue levels and changes.

## 11. Revenue, Cost, Profit, and Margin

Revenue is the value generated from sales; costs are resources consumed; profit is revenue minus relevant costs. Gross profit often subtracts cost of goods sold, while operating profit also accounts for operating expenses.

A margin is profit divided by the relevant revenue base. Distinguish **margin** from **markup**: margin uses selling price as denominator, while markup uses cost. Compare businesses only when accounting definitions are consistent.

In [ ]:
revenue_total = orders["revenue"].sum()  # Add revenue across all transaction lines.
cost_total = orders["cost"].sum()  # Add costs across all transaction lines.
gross_profit_total = revenue_total - cost_total  # Calculate gross profit for this simplified dataset.
gross_margin = gross_profit_total / revenue_total  # Calculate gross profit as a share of revenue.
markup_on_cost = gross_profit_total / cost_total  # Calculate profit as a share of cost.
print(f"Revenue: ${revenue_total:,.2f}")  # Show total revenue.
print(f"Cost: ${cost_total:,.2f}")  # Show total cost.
print(f"Gross profit: ${gross_profit_total:,.2f}")  # Show gross profit in currency units.
print(f"Gross margin: {gross_margin:.1%}")  # Show profit as a share of revenue.
print(f"Markup on cost: {markup_on_cost:.1%}")  # Show profit as a share of cost.

## 12. Unit Economics

Unit economics estimates contribution per unit, customer, order, or subscription. The right unit depends on the business model. For an order, contribution may be net revenue minus product, shipping, payment, and other variable costs.

Positive contribution is not the same as overall profitability: fixed costs still need to be covered. Include all relevant variable costs and be clear whether customer acquisition cost is allocated per acquired customer or per order.

In [ ]:
unit_economics = pd.DataFrame({  # Define illustrative per-order economics.
    "item": ["Average order revenue", "Product cost", "Shipping subsidy", "Payment fees", "Other variable cost"],  # List components.
    "amount": [80.0, 42.0, 5.0, 2.4, 1.6],  # Store amounts in currency units per order.
})  # Finish the unit-economics table.
average_revenue = unit_economics.loc[unit_economics["item"] == "Average order revenue", "amount"].iloc[0]  # Extract average revenue.
variable_costs = unit_economics.loc[unit_economics["item"] != "Average order revenue", "amount"].sum()  # Add all variable cost components.
contribution = average_revenue - variable_costs  # Calculate contribution per order before fixed costs.
contribution_margin = contribution / average_revenue  # Calculate contribution as a share of revenue.
print(unit_economics.to_string(index=False))  # Display per-order economics.
print(f"Variable cost per order: {variable_costs:.2f}")  # Display total variable costs.
print(f"Contribution per order: {contribution:.2f}")  # Display remaining amount per order.
print(f"Contribution margin: {contribution_margin:.1%}")  # Display contribution percentage.

## 13. Break-Even Analysis

Break-even analysis estimates the volume at which contribution covers fixed costs. With constant unit price and variable cost, `break-even units = fixed costs / (price per unit − variable cost per unit)`.

The model assumes a stable price, cost structure, and sales mix. For multiple products, use weighted average contribution or model products separately. Break-even is a planning approximation, not a guarantee.

In [ ]:
fixed_costs = 30000  # Define fixed costs for the period.
selling_price = 75  # Define the price per unit.
variable_cost_per_unit = 45  # Define the variable cost per unit.
contribution_per_unit = selling_price - variable_cost_per_unit  # Calculate the amount each unit contributes to fixed costs.
break_even_units = fixed_costs / contribution_per_unit  # Calculate the theoretical break-even quantity.
break_even_revenue = break_even_units * selling_price  # Convert break-even volume into revenue.
print("Contribution per unit:", contribution_per_unit)  # Display contribution.
print("Break-even units:", int(np.ceil(break_even_units)))  # Round units up to the next whole unit.
print(f"Break-even revenue: ${break_even_revenue:,.2f}")  # Display break-even sales revenue.

## 14. Cost–Benefit Analysis and ROI

Cost–benefit analysis compares expected benefits with costs. ROI is often defined as `(benefit − cost) / cost`, but organizations may use different conventions, so write down the formula and time horizon.

Include implementation cost, ongoing costs, opportunity cost, risks, and the time value of money when relevant. Expected benefits should be linked to measurable assumptions and not double-counted.

In [ ]:
implementation_cost = 24000  # Estimate one-time implementation cost.
annual_operating_cost = 6000  # Estimate operating costs over the first year.
annual_benefit = 42000  # Estimate monetary benefit during the first year.
total_first_year_cost = implementation_cost + annual_operating_cost  # Calculate total first-year cost.
net_benefit = annual_benefit - total_first_year_cost  # Calculate benefit after costs.
roi = net_benefit / total_first_year_cost  # Calculate first-year ROI using net benefit over cost.
payback_months = implementation_cost / ((annual_benefit - annual_operating_cost) / 12)  # Estimate payback assuming benefits and operating costs accrue evenly.
print(f"First-year cost: ${total_first_year_cost:,.0f}")  # Display the first-year total cost.
print(f"First-year net benefit: ${net_benefit:,.0f}")  # Display estimated net benefit.
print(f"First-year ROI: {roi:.1%}")  # Display ROI using the stated definition.
print(f"Approximate payback: {payback_months:.1f} months")  # Display simple payback in months, before discounting and timing nuances.

## 15. Pareto Analysis: Find the Vital Few

Pareto analysis ranks contributors and calculates cumulative share. The 80/20 principle is a heuristic, not a law: a small number of products, issues, or customers may account for a large portion of an outcome.

Use it to find where investigation might have the greatest impact, then validate whether the highest contributors are actually actionable.

In [ ]:
category_profit = orders.groupby("category")["gross_profit"].sum().sort_values(ascending=False)  # Sum profit by category and rank descending.
pareto = category_profit.to_frame("gross_profit")  # Start a report with each category's profit.
pareto["share"] = pareto["gross_profit"] / pareto["gross_profit"].sum()  # Calculate each category's share of total profit.
pareto["cumulative_share"] = pareto["share"].cumsum()  # Calculate the cumulative contribution in ranked order.
print(pareto.round(3))  # Display categories and their cumulative contribution.
important_categories = pareto[pareto["cumulative_share"] <= 0.8].index.tolist()  # Select categories within the 80% threshold where possible.
print("\nCategories fully within first 80%:", important_categories)  # Display the subset to investigate first.

## 16. Root-Cause Analysis: Five Whys

Root-cause analysis seeks underlying process or system causes rather than stopping at the first visible symptom. The Five Whys technique repeatedly asks why an issue occurred until the team reaches a cause that can be tested or acted on.

Do not assume there is always one root cause. Use logs, interviews, process observations, and data to validate hypotheses. Avoid blaming individuals when process design or incentives may be responsible.

In [ ]:
five_whys = pd.DataFrame({  # Record an example chain of questions and hypotheses.
    "level": [1, 2, 3, 4, 5],  # Number each why-question.
    "question": ["Why were orders late?", "Why did packing finish late?", "Why did work queue up?", "Why was capacity insufficient?", "Why was staffing not adjusted?"],  # Write each follow-up question.
    "hypothesis_to_validate": ["Dispatch missed cutoff", "Packing took longer than planned", "Demand peaked in one shift", "Schedule used average demand", "No peak-demand alert existed"],  # Note a hypothesis, not a proven fact.
    "evidence_needed": ["Dispatch timestamps", "Pack-time records", "Hourly order arrivals", "Schedule versus volume", "Alert and planning records"],  # Identify evidence needed to validate it.
})  # Finish the root-cause worksheet.
print(five_whys.to_string(index=False))  # Review the chain and evidence requirements.

## 17. Process Mapping and Bottlenecks

A process map shows the stages through which work moves. A bottleneck is a stage that constrains the throughput of the overall process. The stage with the longest individual task time is not always the bottleneck if parallel capacity or queue sizes differ.

Measure arrival rates, service times, work-in-progress, waiting time, and rework. Process improvements should be checked against end-to-end outcomes, not just the speed of one stage.

In [ ]:
process = pd.DataFrame({  # Create an illustrative process capacity table.
    "stage": ["Order review", "Picking", "Packing", "Dispatch"],  # List process stages in sequence.
    "units_per_hour_capacity": [90, 70, 45, 60],  # Estimate hourly capacity at each stage.
    "average_wait_minutes": [5, 18, 32, 12],  # Record average waiting time at each stage.
})  # Finish the process table.
bottleneck = process.loc[process["units_per_hour_capacity"].idxmin(), "stage"]  # Find the stage with the lowest stated capacity.
process["relative_capacity"] = process["units_per_hour_capacity"] / process["units_per_hour_capacity"].max()  # Compare each capacity with the highest capacity.
print(process.to_string(index=False))  # Display stage capacity and queueing information.
print("\nFirst capacity bottleneck to investigate:", bottleneck)  # Display the lowest-capacity stage; validate with real flow data.

## 18. Measure Process Cycle Time

Cycle time measures elapsed time from a defined start event to a defined end event. Lead time may include the entire time a customer waits, while processing time may include only active work. Definitions vary by organization.

Use timestamps with consistent time zones, remove invalid records transparently, and inspect distributions rather than only averages. Medians and high percentiles often reveal customer experience better than the mean alone.

In [ ]:
rng_cycle = np.random.default_rng(81)  # Create a reproducible random generator.
cycle_minutes = rng_cycle.gamma(shape=2.2, scale=16, size=300)  # Generate positive, right-skewed cycle-time observations.
cycle_summary = {  # Summarize different parts of the cycle-time distribution.
    "mean_minutes": cycle_minutes.mean(),  # Calculate the arithmetic mean.
    "median_minutes": np.median(cycle_minutes),  # Calculate the median.
    "p90_minutes": np.percentile(cycle_minutes, 90),  # Calculate the time below which 90% of observations fall.
    "p95_minutes": np.percentile(cycle_minutes, 95),  # Calculate the time below which 95% of observations fall.
}  # Finish the summary dictionary.
for metric, value in cycle_summary.items():  # Iterate over the calculated metrics.
    print(f"{metric}: {value:.1f}")  # Display each statistic to one decimal place.

## 19. Cohort Analysis and Retention

Cohort analysis groups customers by a shared starting event, such as first purchase month, and tracks behavior over subsequent periods. Retention is the proportion of a cohort that remains active in a later period.

Cohorts separate acquisition timing from customer behavior. Ensure each customer has one clearly defined cohort and use consistent activity definitions across months.

In [ ]:
# Create a small illustrative customer-month activity table.
activity = pd.DataFrame({  # Store customer IDs and the month they were active.
    "customer_id": [1, 1, 1, 2, 2, 3, 4, 4, 5, 5, 5],  # Use repeated IDs for recurring customers.
    "active_month": pd.to_datetime(["2026-01-01", "2026-02-01", "2026-03-01", "2026-01-01", "2026-02-01", "2026-02-01", "2026-02-01", "2026-03-01", "2026-03-01", "2026-04-01", "2026-05-01"]),  # Record month-level activity.
})  # Finish the activity table.
activity["active_month"] = activity["active_month"].dt.to_period("M")  # Convert dates to month periods.
first_month = activity.groupby("customer_id")["active_month"].min().rename("cohort_month")  # Identify each customer's first active month.
activity = activity.join(first_month, on="customer_id")  # Attach cohort month to every customer activity row.
activity["months_since_cohort"] = (activity["active_month"].astype(int) - activity["cohort_month"].astype(int))  # Calculate cohort age in months.
cohort_counts = activity.groupby(["cohort_month", "months_since_cohort"])["customer_id"].nunique().unstack(fill_value=0)  # Count active customers by cohort age.
cohort_size = cohort_counts.iloc[:, 0]  # Take each cohort's size in its first active month.
retention = cohort_counts.div(cohort_size, axis=0)  # Divide active counts by initial cohort size.
print("Cohort active-customer counts:\n", cohort_counts)  # Display the counts by cohort month.
print("\nRetention rates:\n", retention.round(2))  # Display each cohort's retention rate by month since start.

## 20. Churn Rate

Churn measures the loss of customers or subscribers over a defined period. A common customer churn formula is customers lost during the period divided by customers active at the beginning, but business rules differ around reactivation, pauses, and new customers.

Use a consistent population and period. Do not divide lost customers by an end-of-period customer count unless that is the specifically agreed definition.

In [ ]:
customers_at_start = 2400  # Record the number of customers active at the beginning of the month.
customers_lost = 96  # Record customers who meet the agreed churn definition during the month.
new_customers = 180  # Record new customers acquired during the month.
customers_at_end = customers_at_start - customers_lost + new_customers  # Reconcile the ending count under a simplified model.
churn_rate = customers_lost / customers_at_start  # Calculate churn using the opening-customer denominator.
net_customer_growth = (customers_at_end - customers_at_start) / customers_at_start  # Calculate relative net change in customer count.
print(f"Opening customers: {customers_at_start:,}")  # Display the starting base.
print(f"Customer churn rate: {churn_rate:.1%}")  # Display churn rate.
print(f"Ending customers: {customers_at_end:,}")  # Display reconciled ending customers.
print(f"Net customer growth: {net_customer_growth:.1%}")  # Display net growth after acquisitions and losses.

## 21. Customer Lifetime Value (Simple Estimate)

Customer lifetime value (CLV or LTV) estimates the value associated with a customer over a defined relationship. A simplified contribution-based model may use average order contribution × purchase frequency × expected lifetime.

Real CLV often needs retention, discounting, margin, acquisition costs, and uncertainty. Revenue-based LTV can overstate value when costs differ. State the model assumptions and avoid treating a rough estimate as an observed fact.

In [ ]:
average_order_revenue = 90  # Set illustrative revenue per order.
gross_margin_rate = 0.42  # Estimate the gross margin proportion.
orders_per_customer_per_year = 3.5  # Estimate annual purchase frequency.
expected_customer_years = 2.4  # Estimate customer lifespan in years.
contribution_per_order = average_order_revenue * gross_margin_rate  # Estimate gross contribution per order.
simple_ltv = contribution_per_order * orders_per_customer_per_year * expected_customer_years  # Estimate lifetime contribution using constant assumptions.
customer_acquisition_cost = 65  # Estimate the cost to acquire a customer.
ltv_to_cac = simple_ltv / customer_acquisition_cost  # Compare estimated lifetime contribution to acquisition cost.
print(f"Contribution per order: ${contribution_per_order:.2f}")  # Display contribution per order.
print(f"Simple contribution LTV: ${simple_ltv:.2f}")  # Display the modeled lifetime contribution.
print(f"Acquisition cost: ${customer_acquisition_cost:.2f}")  # Display acquisition cost.
print(f"LTV:CAC ratio: {ltv_to_cac:.2f}:1")  # Display the ratio; interpret it with retention and overhead assumptions.

## 22. RFM Customer Analysis

RFM stands for **Recency, Frequency, and Monetary value**. It summarizes how recently a customer purchased, how often they purchased, and how much they spent over a chosen window.

RFM can support CRM prioritization and segment discovery, but it is a heuristic—not a causal model. Define the observation date and window carefully, and consider margins and returns rather than spending alone.

In [ ]:
analysis_date = orders["order_date"].max() + pd.Timedelta(days=1)  # Define the observation date after the latest sample order.
rfm = orders.groupby("customer_id").agg(  # Aggregate transaction history to the customer level.
    last_order=("order_date", "max"),  # Find each customer's most recent order date.
    frequency=("order_id", "nunique"),  # Count distinct orders per customer.
    monetary=("revenue", "sum"),  # Sum transaction-line revenue per customer.
)  # Finish the customer-level aggregation.
rfm["recency_days"] = (analysis_date - rfm["last_order"]).dt.days  # Calculate days since the most recent order.
rfm["recency_score"] = pd.qcut(rfm["recency_days"].rank(method="first"), 4, labels=[4, 3, 2, 1]).astype(int)  # Give more recent customers higher scores.
rfm["frequency_score"] = pd.qcut(rfm["frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)  # Give more frequent customers higher scores.
rfm["monetary_score"] = pd.qcut(rfm["monetary"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)  # Give higher spenders higher scores.
print(rfm.sort_values("monetary", ascending=False).head(10).round(2))  # Show ten customers with the highest transaction revenue.

## 23. Market Sizing: TAM, SAM, and SOM

TAM is the total addressable market, SAM is the serviceable available market for the chosen offering, and SOM is the realistically obtainable share over a defined horizon.

Market sizing is an assumption-driven estimate. Document source, year, geography, customer definition, willingness to pay, and overlap assumptions. A bottom-up calculation based on reachable customers and spend is often easier to audit than a broad top-down percentage.

In [ ]:
total_potential_customers = 500000  # Estimate all potential customers in the broad market.
annual_spend_per_customer = 240  # Estimate annual spend per customer.
serviceable_share = 0.30  # Estimate the proportion that the offering can serve.
obtainable_share_of_serviceable = 0.08  # Estimate the share of the serviceable market achievable within the planning horizon.
tam = total_potential_customers * annual_spend_per_customer  # Calculate total addressable market value.
sam = tam * serviceable_share  # Calculate serviceable available market under the stated assumption.
som = sam * obtainable_share_of_serviceable  # Calculate obtainable market value under the stated assumption.
print(f"TAM: ${tam:,.0f} per year")  # Display the total addressable market estimate.
print(f"SAM: ${sam:,.0f} per year")  # Display the serviceable market estimate.
print(f"SOM: ${som:,.0f} per year")  # Display the obtainable market estimate.
print("Treat these values as scenario estimates until assumptions are supported by evidence.")  # Flag the model as assumption-dependent.

## 24. Competitive Analysis

Competitive analysis compares alternatives on criteria that matter to customers, such as price, quality, availability, trust, support, and convenience. Include indirect substitutes, not just businesses with similar names.

A comparison matrix is useful for structuring evidence, but scores may be subjective. Use a consistent scoring rubric and separate verified facts from assumptions.

In [ ]:
competitors = pd.DataFrame({  # Create an example competitive-positioning matrix.
    "provider": ["Our offer", "Competitor A", "Competitor B"],  # Name the illustrative options.
    "price_score_1_to_5": [4, 3, 5],  # Score relative price attractiveness, where 5 is most attractive.
    "quality_score_1_to_5": [4, 5, 3],  # Score quality using a pre-defined rubric.
    "delivery_score_1_to_5": [5, 3, 4],  # Score delivery convenience.
    "support_score_1_to_5": [3, 4, 2],  # Score customer support.
})  # Finish the comparison matrix.
criteria = ["price_score_1_to_5", "quality_score_1_to_5", "delivery_score_1_to_5", "support_score_1_to_5"]  # List the dimensions used in scoring.
competitors["unweighted_score"] = competitors[criteria].mean(axis=1)  # Calculate an equally weighted mean score.
print(competitors.sort_values("unweighted_score", ascending=False).round(2).to_string(index=False))  # Rank options by the illustrative score.
print("Validate scores with evidence; equal weighting is an assumption, not a fact.")  # State an important limitation.

## 25. Prioritize Requirements with MoSCoW

MoSCoW classifies requirements as **Must have, Should have, Could have, and Won't have for now**. It helps teams manage scope under constraints.

Prioritization should consider user value, risk, effort, dependencies, and deadlines. If everything is marked “Must,” the method has not helped the team make trade-offs.

In [ ]:
requirements = pd.DataFrame({  # Record a simplified set of product requirements.
    "requirement": ["Secure login", "Download monthly report", "Custom chart colors", "AI-generated commentary", "Export to CSV"],  # Name the requirements.
    "business_value": [5, 5, 1, 3, 4],  # Rate value from 1 (low) to 5 (high).
    "effort": [3, 2, 1, 5, 2],  # Estimate effort from 1 (low) to 5 (high).
    "risk_if_missing": [5, 4, 1, 1, 3],  # Rate risk of omission.
    "priority": ["Must", "Must", "Could", "Won't now", "Should"],  # Assign a MoSCoW category.
})  # Finish the requirements list.
requirements["value_per_effort"] = requirements["business_value"] / requirements["effort"]  # Calculate a simple value-to-effort signal.
print(requirements.sort_values(["priority", "value_per_effort"], ascending=[True, False]).to_string(index=False))  # Display requirements and the extra prioritization signal.
print("Use the ratio as a discussion aid, not as an automatic prioritization rule.")  # Explain the limitation of the simple score.

## 26. Risk Assessment Matrix

A risk register records an event, its likelihood, impact, owner, mitigation, and status. A simple risk score is often likelihood × impact, with each rated on an agreed scale.

This score is a prioritization aid, not a precise probability model. High-severity risks may need escalation even when they are unlikely. Include mitigation owners and review dates so the register drives action.

In [ ]:
risks = pd.DataFrame({  # Build a sample business risk register.
    "risk": ["Data access delay", "Low adoption", "Supplier disruption", "Reporting error"],  # Describe each risk.
    "likelihood_1_to_5": [4, 3, 2, 3],  # Rate likelihood on a 1-to-5 scale.
    "impact_1_to_5": [3, 4, 5, 5],  # Rate impact on the same scale.
    "owner": ["Data lead", "Product lead", "Operations", "Analytics lead"],  # Assign an accountable owner.
    "mitigation": ["Confirm access early", "Train users and test usability", "Qualify an alternate supplier", "Add reconciliations and checks"],  # Write a risk response.
})  # Finish the risk register.
risks["risk_score"] = risks["likelihood_1_to_5"] * risks["impact_1_to_5"]  # Calculate the simple risk score.
risks["priority"] = np.select([risks["risk_score"] >= 15, risks["risk_score"] >= 8], ["High", "Medium"], default="Low")  # Convert scores into illustrative priority bands.
print(risks.sort_values("risk_score", ascending=False).to_string(index=False))  # Review the highest-scoring risks first.

## 27. Scenario and Sensitivity Analysis

Scenario analysis compares outcomes under alternative assumptions; sensitivity analysis varies one or more inputs to identify which assumptions matter most. Scenarios may represent base, optimistic, and pessimistic cases.

Label assumptions clearly and avoid presenting a scenario as a forecast. Use ranges where inputs are uncertain, and investigate whether a decision remains sensible across plausible outcomes.

In [ ]:
base_units = 12000  # Estimate baseline units sold.
base_price = 50  # Set base selling price.
base_variable_cost = 31  # Set base variable cost per unit.
fixed_cost = 90000  # Set fixed cost for the period.
scenarios = pd.DataFrame({  # Define alternative assumptions for volume and pricing.
    "scenario": ["Downside", "Base", "Upside"],  # Name the three scenarios.
    "volume_change": [-0.15, 0.00, 0.18],  # Assume volume changes by scenario.
    "price_change": [-0.03, 0.00, 0.02],  # Assume unit price changes by scenario.
})  # Finish scenario assumptions.
scenarios["units"] = base_units * (1 + scenarios["volume_change"])  # Calculate units under each scenario.
scenarios["price"] = base_price * (1 + scenarios["price_change"])  # Calculate price under each scenario.
scenarios["revenue"] = scenarios["units"] * scenarios["price"]  # Calculate scenario revenue.
scenarios["profit"] = scenarios["units"] * (scenarios["price"] - base_variable_cost) - fixed_cost  # Calculate a simplified contribution-based profit.
print(scenarios.round(2).to_string(index=False))  # Compare the scenario outcomes.
print("Change one assumption at a time to understand sensitivity.")  # Suggest a follow-up analysis.

## 28. Forecast with a Moving Average

A moving average smooths short-term fluctuations by averaging a fixed number of recent observations. It is a useful baseline, but it can lag when there is a trend and miss annual seasonality.

Forecasting requires a clear horizon and time grain. Evaluate forecasts on a holdout period and compare them with a naive or seasonal-naive baseline before relying on them for planning.

In [ ]:
monthly_sales = pd.Series([82, 85, 87, 91, 90, 98, 101, 99, 104, 108, 110, 118], index=pd.period_range("2025-01", periods=12, freq="M"), name="sales")  # Create twelve months of illustrative sales.
moving_average = monthly_sales.rolling(window=3).mean()  # Calculate a trailing three-month average for each month.
forecast_next_month = monthly_sales.iloc[-3:].mean()  # Use the last three months' average as a simple next-month forecast.
forecast_report = pd.DataFrame({"actual": monthly_sales, "three_month_average": moving_average})  # Combine actual values and smoothed values.
print(forecast_report.round(2))  # Display actual sales and the trailing average.
print(f"Simple forecast for next month: {forecast_next_month:.2f}")  # Display the simple baseline forecast.

## 29. Evaluate an A/B Test

An A/B test compares outcomes between randomly assigned variants. Business interpretation should include baseline rate, treatment rate, absolute percentage-point change, relative lift, uncertainty, and the potential value of the change.

A p-value alone is insufficient. Check randomization, sample-ratio mismatch, eligibility, duplicate users, exposure definitions, guardrail metrics, and whether the experiment ran for a preplanned duration.

In [ ]:
users_a, conversions_a = 1500, 120  # Store the number of users and conversions in variant A.
users_b, conversions_b = 1480, 135  # Store the number of users and conversions in variant B.
rate_a = conversions_a / users_a  # Calculate the control conversion rate.
rate_b = conversions_b / users_b  # Calculate the treatment conversion rate.
absolute_lift = rate_b - rate_a  # Calculate the absolute percentage-point change.
relative_lift = absolute_lift / rate_a if rate_a else np.nan  # Calculate relative lift and handle a zero baseline.
pooled = (conversions_a + conversions_b) / (users_a + users_b)  # Calculate the pooled rate under the null hypothesis.
se_null = np.sqrt(pooled * (1 - pooled) * (1 / users_a + 1 / users_b))  # Calculate the null standard error.
z = absolute_lift / se_null  # Standardize the observed difference.
p_value = 2 * stats.norm.sf(abs(z))  # Calculate the two-sided z-test p-value.
print(f"Control conversion: {rate_a:.2%}")  # Display control rate.
print(f"Treatment conversion: {rate_b:.2%}")  # Display treatment rate.
print(f"Absolute lift: {absolute_lift:.2%} percentage points")  # Display absolute lift.
print(f"Relative lift: {relative_lift:.1%}")  # Display relative lift.
print(f"Approximate p-value: {p_value:.4f}")  # Display the test result.
print("Check test design and commercial impact before deciding to launch.")  # Add a decision-quality reminder.

## 30. Write Requirements and Acceptance Criteria

Business analysts turn needs into clear, testable requirements. A user story often follows “As a [user], I want [capability], so that [benefit].” Acceptance criteria define observable conditions that must be met.

Good requirements are specific, testable, traceable to a business objective, and understandable by business and technical teams. Avoid prescribing a technical solution too early unless it is a genuine constraint.

In [ ]:
user_story = "As a sales manager, I want a weekly region-level revenue report so I can identify underperforming regions."  # State the need from a user's perspective.
acceptance_criteria = [  # Define observable outcomes that can be verified.
    "The report includes all approved regions.",  # Define expected coverage.
    "Revenue uses the finance-approved net-revenue definition.",  # Specify the agreed metric.
    "Users can filter by week and sales channel.",  # Define required interactions.
    "The displayed totals reconcile with the source system within the agreed tolerance.",  # Define a data-quality condition.
    "The report shows the refresh timestamp.",  # Make data freshness visible.
]  # Finish the acceptance criteria.
print("User story:", user_story)  # Display the user story.
print("\nAcceptance criteria:")  # Label the criteria list.
for criterion in acceptance_criteria:  # Loop through each testable condition.
    print("☐", criterion)  # Print an unchecked acceptance criterion.

## 31. Turn Analysis into Recommendations

A useful recommendation connects evidence to an action, expected impact, effort, risk, and a way to measure success. Separate findings from explanations: a pattern in the data may suggest a cause, but further testing may be needed to establish it.

Strong recommendations are specific and decision-oriented. State what should happen, why it matters, how success will be measured, who owns the next step, and what uncertainty remains.

In [ ]:
recommendation = {  # Draft an example executive recommendation.
    "finding": "Packing has the lowest stated hourly capacity and the longest average wait",  # Summarize the evidence from the process analysis.
    "action": "Run a two-week staffing and workflow pilot at packing",  # Recommend a bounded action.
    "success_metric": "Median order-to-dispatch time and on-time dispatch rate",  # Define measurable outcomes.
    "guardrail": "Packing error rate must not increase",  # Protect quality while improving speed.
    "owner": "Operations Lead",  # Assign responsibility.
    "uncertainty": "Capacity figures are estimates and should be validated with actual flow data",  # State what is not yet known.
}  # Finish the recommendation.
for key, value in recommendation.items():  # Iterate through the recommendation components.
    print(f"{key.replace('_', ' ').title()}: {value}")  # Print a concise draft that can be copied into a report.

## 32. Executive Summary and Decision Dashboard

An executive summary should answer: What happened? Why does it matter? What is driving it? What action is recommended? What are the risks and next steps?

A dashboard is not a substitute for analysis. Use a small number of decision-relevant KPIs, display definitions and time periods, show useful comparisons, and make data freshness and limitations visible.

In [ ]:
# Create a compact executive KPI summary from the sample orders.
executive_summary = pd.DataFrame({  # Build a one-row management summary.
    "revenue": [orders["revenue"].sum()],  # Calculate total revenue.
    "gross_profit": [orders["gross_profit"].sum()],  # Calculate total gross profit.
    "unique_orders": [orders["order_id"].nunique()],  # Count distinct orders rather than transaction lines.
    "customers": [orders["customer_id"].nunique()],  # Count distinct customers in the sample.
})  # Finish the summary DataFrame.
executive_summary["gross_margin"] = executive_summary["gross_profit"] / executive_summary["revenue"]  # Calculate gross margin.
executive_summary["average_order_value"] = executive_summary["revenue"] / executive_summary["unique_orders"]  # Calculate revenue per distinct order.
print(executive_summary.T.rename(columns={0: "Value"}).round(2))  # Display each KPI vertically for an executive summary.
print("\nBefore publishing, verify metric definitions, period comparisons, filters, and source reconciliation.")  # Add a final reporting check.